# Session 02 Practice: 2D LiDAR Occupancy Grid Mapping & Costmap Inflation
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 1 | Session 2** (Practice Exercise)  

---

### 🎯 Learning Objectives
1. Implement a **2D Log-Odds Occupancy Grid** mapping pipeline from first principles.
2. Build an **Inverse Sensor Model (ISM)** for range-bearing LiDAR beams with sub-grid ray traversal.
3. Perform sequential Bayesian log-odds updates ($l_{\text{free}}, l_{\text{occ}}$) and clamping across a multi-turn trajectory.
4. Convert log-odds back into probabilistic occupancy $P(m_i = 1 | z_{1:t})$ via the sigmoid function.
5. Generate an inflated **Safety Costmap** with lethal, inscribed, and exponential safety decay layers using **Euclidean Distance Transforms (EDT)**.
6. Visualize interactive 2D spatial maps, cross-sectional slices, and 3-panel perception-to-planning comparison figures with **Plotly**.

---

### 📐 Mathematical Formulation

#### 1. Binary Bayes Filter in Log-Odds Space
To prevent numerical underflow from multiplying small probabilities and simplify recursive Bayesian updates, probability $p \in (0, 1)$ is transformed into log-odds $l \in (-\infty, \infty)$:

$$l = \ln\left(\frac{p}{1 - p}\right) \iff p = \frac{1}{1 + e^{-l}}$$

For prior probability $P(m_{i = 1}) = 0.5$, the initial log-odds $l_0 = \ln(0.5 / 0.5) = 0$.

For each LiDAR beam originating from vehicle pose $(x_s, y_s, \theta)$ with measured range $r$ and beam bearing $\theta_{\text{beam}}$:
1. **Free Cells along ray** ($d < r - \frac{\text{thickness}}{2}$):
   $$l_{t, i} = l_{t-1, i} + l_{\text{free}}, \quad l_{\text{free}} = \ln\left(\frac{P_{\text{free}}}{1 - P_{\text{free}}}\right) < 0$$
2. **Occupied Cell at hit point** ($|d - r| \le \frac{\text{thickness}}{2}$):
   $$l_{t, i} = l_{t-1, i} + l_{\text{occ}}, \quad l_{\text{occ}} = \ln\left(\frac{P_{\text{occ}}}{1 - P_{\text{occ}}}\right) > 0$$
3. **Log-Odds Clamping:**
   $$l_{t, i} \leftarrow \text{clip}(l_{t, i}, l_{\min}, l_{\max})$$
   Clamping prevents overconfidence and enables fast recovery when previously occupied spaces clear or sensor noise occurs.

#### 2. Inflated Safety Costmap via Euclidean Distance Transform (EDT)
Downstream trajectory planners require continuous cost potentials rather than raw probabilities. Using the Euclidean distance $d(x, y)$ from each cell to the nearest lethal obstacle cell ($P \ge P_{\text{thresh}}$):
1. **Lethal Zone** ($P \ge P_{\text{thresh}}$): $\text{Cost} = 254$
2. **Inscribed Vehicle Safety Zone** ($d \le r_{\text{inscribed}}$): $\text{Cost} = 253$ (guaranteed physical collision with vehicle footprint)
3. **Exponential Decay Buffer Zone** ($r_{\text{inscribed}} < d \le r_{\text{inflation}}$):
   $$\text{Cost}(d) = \left\lfloor 253 \cdot e^\left(-\alpha \cdot (d - r_{\text{inscribed}})\right) \right\rfloor$$
4. **Free Drivable Space** ($d > r_{\text{inflation}}$): $\text{Cost} = 0$


In [ ]:
import math
import numpy as np
import polars as pl
from scipy.ndimage import distance_transform_edt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"


## 1. Occupancy Grid & Costmap Inflation Implementation

In this section, you will implement the core components of a 2D LiDAR Occupancy Grid and Costmap Inflation pipeline inside `OccupancyGrid2D`.

---

### 🗺️ Intuitive Guide: Understanding `world_to_grid` (Without Heavy Math)

Why do we need coordinate transformation in autonomous driving?

1. **Continuous Real World vs. Discrete Computer Memory:**
   - Onboard sensors (such as LiDAR scanners, GNSS-RTK receivers, and IMUs) report measurements in continuous physical space (meters). For example, a laser beam detects a street barrier at $(x = 12.43\text{ m}, y = 3.12\text{ m})$ relative to the vehicle coordinate frame.
   - However, computer memory stores spatial maps as **discrete 2D matrix arrays** (tables of rows and columns, like pixels in an image). A matrix only understands integer indices like `map[row, col]`.

2. **The Graph Paper Analogy:**
   - Imagine placing a giant sheet of square grid graph paper over a city intersection. Each square on the paper represents a small patch of real ground (e.g., $20\text{ cm} \times 20\text{ cm}$ cell resolution).
   - To figure out which specific square on our paper map corresponds to a physical obstacle:
     - **Step 1: Align with the Map Corner (Offset):** Real-world coordinates can be negative (e.g. from $-5\text{ m}$ to $+28\text{ m}$). But arrays always index from zero. So we first measure how far the point is from the map's minimum boundary corner (the bottom-left edge). This gives the positive offset distance within the map.
     - **Step 2: Count the Grid Squares:** We divide that physical distance by the width of one grid square. This counts how many squares across and how many squares up the object is located. Because an object sits inside a single discrete square, we drop the fractional remainder (convert to integer) to get the exact row and column.
     - **Step 3: Boundary Safety Check:** Just like dropping a pin on a paper map, if an obstacle is located completely outside the borders of our map sheet, we cannot record it in our array! We must verify that the column is between 0 and `width - 1`, and the row is between 0 and `height - 1`. If the point falls outside, we safely return `None` to prevent index errors.

---

### 📋 Methods to Implement (Student Task)
1. `world_to_grid(x, y)`: Continuous $(x, y) \to$ Discrete $(\text{row}, \text{col})$ index conversion with bounds checking.
2. `update_scan(...)`: Integrates LiDAR beams into the log-odds grid via raycasting and measurement models.
3. `get_probability_map()`: Converts internal log-odds matrix into normalized probabilities $P(m) \in [0.0, 1.0]$.
4. `compute_costmap(...)`: Produces an inflated safety costmap with lethal (254), inscribed (253), and decay zones (1..252).


In [ ]:
class OccupancyGrid2D:
    def __init__(self, x_min: float = -5.0, x_max: float = 28.0,
                 y_min: float = -15.0, y_max: float = 15.0,
                 resolution: float = 0.2):
        """
        2D Occupancy Grid Map representing local vehicle surroundings.

        Real-Life Source Mapping:
        -------------------------
        - x_min, x_max, y_min, y_max: Physical spatial boundaries (in meters) of the
          autonomous vehicle's local sensing envelope / perception horizon.
        - resolution: Physical ground footprint of each grid cell (0.2 m = 20 cm per cell).
          This resolution is standard for autonomous driving to detect curbs, lane edges,
          and pedestrians while maintaining fast real-time compute.
        - self.log_odds: Log-odds occupancy matrix initialized to 0.0.
          Real-life source: Prior probability P = 0.5 (complete uncertainty / unobserved space
          before sensors take any measurements).
        - self.l_occ: Measurement model update when a laser beam reflects off an obstacle (~ +1.734).
          Real-life source: Sensor datasheet detection confidence (~85% true positive rate upon laser return).
        - self.l_free: Measurement model update along the unobstructed laser ray path (~ -0.847).
          Real-life source: Sensor line-of-sight clear confidence (~30% residual occupancy probability).
        - self.l_min, self.l_max: Saturation clamps [-4.0, +4.0] (corresponding to ~0.018 and ~0.982 probability).
          Real-life source: Prevents probability saturation so dynamic obstacles (e.g. pedestrian walking away)
          can be cleared quickly by new observations.
        - self.thickness: Physical laser footprint / obstacle boundary thickness (0.3 m).
        """
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.resolution = resolution

        self.width = int((x_max - x_min) / resolution)
        self.height = int((y_max - y_min) / resolution)

        # Log-odds array initialized to 0.0 (uniform prior P = 0.5: unmapped space)
        self.log_odds = np.zeros((self.height, self.width), dtype=np.float32)

        # Sensor measurement model parameters (from sensor calibration & datasheet)
        self.l_occ = math.log(0.85 / (1.0 - 0.85))  # ~ +1.734 (Obstacle detection echo)
        self.l_free = math.log(0.30 / (1.0 - 0.30))  # ~ -0.847 (Clear laser line-of-sight)
        self.l_min = -4.0  # Minimum log-odds clamp (~0.018 probability)
        self.l_max = 4.0  # Maximum log-odds clamp (~0.982 probability)
        self.thickness = 0.3  # Obstacle boundary thickness in meters

    def world_to_grid(self, x: float, y: float):
        """
        Converts continuous world coordinates (x, y) to discrete grid indices (row, col).
        
        Real-Life Source:
            - x, y: Global coordinate position of vehicle or detected obstacle hit in meters,
              derived from GNSS/IMU localization and raw sensor measurements.

        Why this operation is performed (without math):
            1. Real-world sensor coordinates can be negative and centered anywhere, but computer
               matrices only start at row 0, column 0. We first measure the physical offset of (x, y)
               from the map's minimum corner (x_min, y_min).
            2. We count how many grid squares fit inside that offset distance by dividing by resolution,
               and take the integer portion because the object lands inside a single square cell.
            3. If the coordinate falls outside the map boundaries, return None to avoid reading/writing
               invalid memory. Otherwise, return (row, col).

        Returns:
            (row, col) tuple of integers if inside grid bounds, else None
        """
        # =========================================================================
        # TODO (Student): Implement Coordinate Transformation
        # =========================================================================
        # >>> YOUR CODE HERE >>>
        raise NotImplementedError("Implement world_to_grid!")
        # <<< YOUR CODE HERE <<<

    def grid_to_world(self, row: int, col: int):
        """
        Converts discrete grid indices (row, col) to continuous world coordinates (x, y).

        Real-Life Source:
            - row, col: Discrete matrix cell indices representing an occupied, free, or
              planned waypoint cell in the occupancy grid map.

        Why this operation is performed (without math):
            1. Path planners and search algorithms (like A* or Dijkstra) operate on discrete
               cells/nodes in a matrix, but vehicle controllers (steering, throttle, trajectory
               trackers) require real-world spatial coordinates in meters to steer the vehicle.
            2. A single discrete cell spans an entire square patch of the world. To give the
               controller an unambiguous target, we locate the exact spatial center of that square
               by stepping from the map's minimum physical corner by the cell indices plus a half-step
               (half a cell's width/height).
            3. If the given indices fall outside the allocated grid dimensions, return None to
               prevent referencing non-existent physical space. Otherwise, return (x, y).

        Returns:
            (x, y) tuple of floats representing the cell center in world meters if inside
            grid bounds, else None
        """
        # =========================================================================
        # TODO (Student): Implement Coordinate Transformation
        # =========================================================================
        # >>> YOUR CODE HERE >>>
        raise NotImplementedError("Implement grid_to_world!")
        # <<< YOUR CODE HERE <<<

    def update_scan(self, robot_x: float, robot_y: float, robot_theta: float,
                    ranges: np.ndarray, angles: np.ndarray, max_range: float = 15.0):
        """
        Updates the log-odds grid with one complete LiDAR scan from the vehicle.

        Real-Life Source Mapping:
        -------------------------
        - robot_x, robot_y, robot_theta: Vehicle pose estimated by onboard localization
          (sensor fusion of GNSS-RTK, IMU, and wheel odometry).
        - ranges: Array of distance measurements in meters (Time-of-Flight laser returns).
        - angles: Array of azimuth angles in radians relative to vehicle heading.
        - max_range: Hardware maximum range limit from LiDAR sensor datasheet (e.g. 15.0 m).

        Why this operation is performed (without math):
        ----------------------------------------------
        When a LiDAR fires a laser pulse:
        1. The laser beam travels outward from the vehicle in a straight line through the air.
        2. All the space that the light traveled through before reaching the target is confirmed
           to be EMPTY (free space), because if there were an obstacle, the beam would have stopped earlier.
           Therefore, we decrease the occupancy log-odds along the ray path by self.l_free.
        3. At the point where the laser beam hits an object and bounces back, an obstacle is present.
           Therefore, we increase the occupancy log-odds of that specific hit cell by self.l_occ.
        4. If the laser beam did not hit anything within max_range, the entire beam path is free space,
           and no obstacle hit cell is updated.
        5. Finally, we clamp the map values between self.l_min and self.l_max so probabilities stay bounded.

        TODO (Student):
        For each LiDAR beam:
        1. Compute global beam angle: beam_angle = robot_theta + angle
        2. Determine ray length: ray_len = min(r, max_range), check is_valid_hit = (r < max_range)
        3. Raycast along beam using step_size = self.resolution * 0.4:
           - px = robot_x + dist * cos(beam_angle)
           - py = robot_y + dist * sin(beam_angle)
           - If dist < (r - self.thickness / 2.0), update cell: self.log_odds[row, col] += self.l_free
        4. If is_valid_hit, update obstacle hit cell at distance r with self.l_occ
        5. Clamp self.log_odds within [self.l_min, self.l_max] using np.clip
        """
        # =========================================================================
        # TODO (Student): Implement LiDAR Inverse Sensor Model & Raycasting
        # =========================================================================
        # >>> YOUR CODE HERE >>>
        raise NotImplementedError("Implement update_scan!")
        # <<< YOUR CODE HERE <<<

    def get_probability_map(self) -> np.ndarray:
        """
        Converts the internal log-odds grid into an occupancy probability matrix P(m) in [0.0, 1.0].
        
        Real-Life Source:
            Downstream trajectory planners and obstacle avoidance modules cannot directly consume
            abstract log-odds values. They require intuitive probabilities (e.g. 0.90 = 90% chance of collision).

        Why this operation is performed (without math):
            We use log-odds during sensor updates because simple addition and subtraction update the grid
            rapidly without floating-point underflow. But when outputting the map for planning and display,
            we pass the values through an S-shaped curve (sigmoid) that smoothly compresses (-inf, +inf)
            into the real-world probability range [0.0, 1.0], where 0 log-odds becomes exactly 50% probability.

        TODO (Student):
        Apply logistic sigmoid: P = 1 / (1 + exp(-log_odds))
        """
        # =========================================================================
        # TODO (Student): Implement Log-Odds to Probability Conversion
        # =========================================================================
        # >>> YOUR CODE HERE >>>
        raise NotImplementedError("Implement get_probability_map!")
        # <<< YOUR CODE HERE <<<

    def compute_costmap(self, prob_threshold: float = 0.65,
                        r_inscribed: float = 1.0,
                        r_inflation: float = 2.5,
                        decay_factor: float = 2.0) -> np.ndarray:
        """
        Computes an inflated safety costmap from the estimated occupancy grid.

        Real-Life Source Mapping:
        -------------------------
        - Vehicle Geometry & Safety Envelope: Real cars are not infinitesimal points—they have length,
          width, and turning overhangs. A motion planner that treats the car as a point would scrape
          bumpers against walls.
        - prob_threshold: Confidence threshold (e.g. 0.65 = 65% probability) above which a cell is
          treated as a genuine physical obstacle.
        - r_inscribed: Radius of the vehicle's inscribed circle in meters (e.g. 1.0 m). Any point within
          this distance from an obstacle guarantees physical body collision regardless of heading!
        - r_inflation: Safety margin buffer radius (e.g. 2.5 m) to keep comfortable clearance.
        - decay_factor: Exponential decay rate determining how quickly repulsive cost drops off.

        Cost Layer Values:
        ------------------
        - 254: Lethal Obstacle (Direct obstacle cell where P >= prob_threshold)
        - 253: Inscribed Vehicle Radius (Guaranteed vehicle body collision zone)
        - 1..252: Exponential Safety Decay Zone: 253 * exp(-decay_factor * (dist - r_inscribed))
                  Acts like a repulsive magnetic field guiding planners toward lane centers.
        - 0: Free / Safe Drivable Space

        TODO (Student):
        1. Identify lethal obstacle mask: lethal_mask = (prob_map >= prob_threshold)
        2. Compute Euclidean distance from each cell to nearest lethal cell in meters:
           distance_grid = distance_transform_edt(~lethal_mask) * self.resolution
        3. Fill costmap array (dtype=np.uint8):
           - Set lethal_mask cells to 254
           - Set non-lethal cells with distance <= r_inscribed to 253
           - Set decay zone cells (r_inscribed < dist <= r_inflation) to 253 * exp(-decay_factor * (dist - r_inscribed))
        """
        # =========================================================================
        # TODO (Student): Implement Multi-Layer Inflated Safety Costmap
        # =========================================================================
        # >>> YOUR CODE HERE >>>
        raise NotImplementedError("Implement compute_costmap!")
        # <<< YOUR CODE HERE <<<


## 2. Synthetic Multi-Pose LiDAR Telemetry Generator

We construct a multi-turn vehicle trajectory navigating around a complex obstacle arrangement:
- **Foreground Box 1:** Located at $[7, 11] \times [-2.5, 2.5]$ m, directly occluding Box 2 from the initial pose.
- **Occluded Box 2:** Located at $[16, 19.5] \times [-2.0, 2.0]$ m in the shadow of Box 1.
- **Trajectory:** 16 sequential vehicle poses undergoing 4 directional changes (East $\to$ North $\to$ East $\to$ South $\to$ West), progressively revealing the occluded obstacle.


In [ ]:
def get_default_obstacle_boxes():
    """Returns ground truth obstacle box bounds [(x0, x1, y0, y1), ...]."""
    return [
        (7.0, 11.0, -2.5, 2.5),  # Foreground obstacle (blocks central path)
        (16.0, 19.5, -2.0, 2.0),  # Occluded obstacle (shadowed until vehicle navigates around)
        (6.0, 10.0, 6.0, 9.5),  # North obstacle
        (6.0, 10.0, -9.5, -6.0),  # South obstacle
        (21.0, 25.0, 2.5, 6.5),  # Far East obstacle
    ]


def create_ground_truth_map(grid: OccupancyGrid2D, boxes: list) -> np.ndarray:
    """Creates a discrete binary Ground Truth matrix (height x width)."""
    gt_map = np.zeros((grid.height, grid.width), dtype=np.uint8)
    for (x0, x1, y0, y1) in boxes:
        col_min = max(0, int((x0 - grid.x_min) / grid.resolution))
        col_max = min(grid.width, int((x1 - grid.x_min) / grid.resolution) + 1)
        row_min = max(0, int((y0 - grid.y_min) / grid.resolution))
        row_max = min(grid.height, int((y1 - grid.y_min) / grid.resolution) + 1)
        gt_map[row_min:row_max, col_min:col_max] = 1
    return gt_map


def generate_synthetic_lidar_dataset(boxes: list = None, num_scans: int = 16) -> pl.DataFrame:
    """Generates synthetic LiDAR telemetry data using Polars DataFrame across 4 turns."""
    if boxes is None:
        boxes = get_default_obstacle_boxes()

    waypoints = [
        (0.0, 0.0, 0.0),
        (2.5, 0.0, 0.0),
        (5.0, 0.0, 0.0),  # Turn 1: Steer North (+90 deg)
        (5.0, 2.2, math.pi / 2.0),
        (5.0, 4.5, math.pi / 2.0),  # Turn 2: Steer East (0 deg)
        (8.0, 4.5, 0.0),
        (11.0, 4.5, 0.0),
        (13.5, 4.5, 0.0),  # Turn 3: Steer South (-90 deg)
        (13.5, 2.0, -math.pi / 2.0),
        (13.5, 0.0, -math.pi / 2.0),  # Directly between both obstacles (fully unoccluded)
        (13.5, -2.0, -math.pi / 2.0),
        (13.5, -4.5, -math.pi / 2.0),  # Turn 4: Steer West (180 deg)
        (11.0, -4.5, math.pi),
        (8.0, -4.5, math.pi),
        (5.0, -4.5, math.pi),
        (2.5, -4.5, math.pi),
    ]
    waypoints = waypoints[:num_scans]
    records = []
    num_beams = 180
    angles = np.linspace(-math.pi, math.pi, num_beams)

    for scan_id, (rx, ry, rtheta) in enumerate(waypoints):
        for x0, x1, y0, y1 in boxes:
            assert not (x0 <= rx <= x1 and y0 <= ry <= y1), f"Vehicle collided with box [{x0}, {x1}, {y0}, {y1}]"

        for a in angles:
            beam_dir = rtheta + a
            measured_r = 15.0
            for x0, x1, y0, y1 in boxes:
                for step_r in np.linspace(0.1, 15.0, 300):
                    px = rx + step_r * math.cos(beam_dir)
                    py = ry + step_r * math.sin(beam_dir)
                    if x0 <= px <= x1 and y0 <= py <= y1:
                        measured_r = min(measured_r, step_r)
                        break
            if measured_r < 15.0:
                measured_r += float(np.random.normal(0, 0.03))
            records.append({
                "scan_id": scan_id,
                "robot_x": rx,
                "robot_y": ry,
                "robot_theta": rtheta,
                "beam_angle": float(a),
                "measured_range": float(measured_r)
            })
    return pl.DataFrame(records)


# Generate synthetic dataset
boxes = get_default_obstacle_boxes()
df_lidar = generate_synthetic_lidar_dataset(boxes=boxes, num_scans=16)
print(
    f"Generated LiDAR dataset with {df_lidar.height} rows across {df_lidar['scan_id'].n_unique()} scans (4 direction changes).")
df_lidar.head()


## 3. Sequential Grid Updating & Interactive Visualizations

In [ ]:
# 1. Sequential Scan Processing & 2D Spatial Heatmap
boxes = get_default_obstacle_boxes()
if "df_lidar" not in locals() or df_lidar is None:
    df_lidar = generate_synthetic_lidar_dataset(boxes=boxes, num_scans=16)

grid = OccupancyGrid2D(x_min=-5.0, x_max=28.0, y_min=-15.0, y_max=15.0, resolution=0.2)
unique_scans = df_lidar["scan_id"].unique().sort().to_list()
robot_poses_x = []
robot_poses_y = []

for s_id in unique_scans:
    scan_df = df_lidar.filter(pl.col("scan_id") == s_id)
    rx = scan_df["robot_x"][0]
    ry = scan_df["robot_y"][0]
    rtheta = scan_df["robot_theta"][0]
    ranges = scan_df["measured_range"].to_numpy()
    angles = scan_df["beam_angle"].to_numpy()

    robot_poses_x.append(rx)
    robot_poses_y.append(ry)

    grid.update_scan(rx, ry, rtheta, ranges, angles)

p_map = grid.get_probability_map()

# 2D Spatial Heatmap with Multi-Turn Trajectory Overlay
x_coords = np.linspace(grid.x_min, grid.x_max, grid.width)
y_coords = np.linspace(grid.y_min, grid.y_max, grid.height)

fig_map = go.Figure()

fig_map.add_trace(go.Heatmap(
    z=p_map,
    x=x_coords,
    y=y_coords,
    colorscale="Viridis",
    zmin=0.0,
    zmax=1.0,
    colorbar=dict(title="P(Occupied)")
))

fig_map.add_trace(go.Scatter(
    x=robot_poses_x,
    y=robot_poses_y,
    mode="markers+lines",
    marker=dict(size=10, color="magenta", symbol="triangle-right"),
    line=dict(color="magenta", width=2, dash="dash"),
    name="Vehicle Trajectory (4 Turns)"
))

# Ground truth obstacle boxes for reference
for x0, x1, y0, y1 in boxes:
    fig_map.add_shape(
        type="rect",
        x0=x0, y0=y0, x1=x1, y1=y1,
        line=dict(color="lime", width=2),
        fillcolor="rgba(0, 255, 0, 0.1)"
    )

fig_map.update_layout(
    title="2D LiDAR Occupancy Grid (Multi-Turn Trajectory Resolving Occlusion)",
    xaxis_title="Global X (m)",
    yaxis_title="Global Y (m)",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=900,
    height=600
)
fig_map.show()


In [ ]:
# 2. 1D Cross-Section Slice & Log-Odds Transformation Curves
fig_diag = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Occupancy Probability Cross-Section at Y = 3.5 m",
        "Log-Odds to Probability Sigmoid Function"
    )
)

target_y = 3.5
row_idx = int((target_y - grid.y_min) / grid.resolution)
p_slice = p_map[row_idx, :]

fig_diag.add_trace(
    go.Scatter(x=x_coords, y=p_slice, mode="lines+markers",
               line=dict(color="#1f77b4", width=3),
               marker=dict(size=4),
               name="P(Occupied) Slice"),
    row=1, col=1
)
fig_diag.add_hline(y=0.5, line_dash="dot", line_color="gray", annotation_text="Prior P=0.5", row=1, col=1)
fig_diag.add_hline(y=0.7, line_dash="dash", line_color="red", annotation_text="Occupied Threshold", row=1, col=1)

l_vals = np.linspace(-6, 6, 200)
p_vals = 1.0 / (1.0 + np.exp(-l_vals))

fig_diag.add_trace(
    go.Scatter(x=l_vals, y=p_vals, mode="lines",
               line=dict(color="#2ca02c", width=3),
               name="P = 1 / (1 + e^-l)"),
    row=1, col=2
)
fig_diag.add_vline(x=0.0, line_dash="dot", line_color="gray", row=1, col=2)
fig_diag.add_hline(y=0.5, line_dash="dot", line_color="gray", row=1, col=2)

fig_diag.update_layout(
    title="Grid Mapping Diagnostics: Spatial Cross-Section & Probability Mapping",
    height=450,
    showlegend=False
)
fig_diag.show()


In [ ]:
# 3. Full Perception-to-Planning Pipeline: True Map vs Occupancy Grid vs Inflated Costmap
gt_map = create_ground_truth_map(grid, boxes)
costmap = grid.compute_costmap(prob_threshold=0.65, r_inscribed=1.0, r_inflation=2.5, decay_factor=2.0)

fig_pipeline = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "1. True Map (Ground Truth Obstacles)",
        "2. Estimated Occupancy Grid P(m)",
        "3. Inflated Safety Costmap (Planning)"
    )
)

# Panel 1: Ground Truth
fig_pipeline.add_trace(
    go.Heatmap(z=gt_map, x=x_coords, y=y_coords, colorscale="Greys", zmin=0, zmax=1,
               colorbar=dict(title="True Map", x=0.28)),
    row=1, col=1
)

# Panel 2: Occupancy Probability
fig_pipeline.add_trace(
    go.Heatmap(z=p_map, x=x_coords, y=y_coords, colorscale="Viridis", zmin=0.0, zmax=1.0,
               colorbar=dict(title="P(Occ)", x=0.62)),
    row=1, col=2
)

# Panel 3: Inflated Costmap
fig_pipeline.add_trace(
    go.Heatmap(z=costmap, x=x_coords, y=y_coords, colorscale="Hot_r", zmin=0, zmax=254,
               colorbar=dict(title="Cost", x=1.0)),
    row=1, col=3
)

fig_pipeline.update_layout(
    title="Session 02: End-to-End Environmental Mapping & Costmap Inflation Pipeline",
    width=1350,
    height=500,
    showlegend=False
)
fig_pipeline.update_xaxes(title_text="Global X (m)", scaleanchor="y", scaleratio=1)
fig_pipeline.update_yaxes(title_text="Global Y (m)")
fig_pipeline.show()


## 4. Verification & Validation Unit Tests

In [ ]:
# Test 1: Prior Uniform Probability Initialization
empty_grid = OccupancyGrid2D()
assert np.allclose(empty_grid.get_probability_map(), 0.5), "Initial probability map must be uniformly 0.5!"

# Test 2: Coordinate Transformation Invariance
test_row, test_col = grid.world_to_grid(5.0, 5.0)
assert test_row is not None and test_col is not None, "Valid world coordinate failed to map to grid!"
wx, wy = grid.grid_to_world(test_row, test_col)
assert abs(wx - 5.0) < grid.resolution and abs(wy - 5.0) < grid.resolution, "Grid to world transformation error!"

# Test 3: Occupancy & Free Space Cell Accumulation
occ_count = np.sum(p_map > 0.7)
free_count = np.sum(p_map < 0.3)
print(f"Mapped Statistics: {occ_count} occupied cells (P > 0.7), {free_count} free cells (P < 0.3).")
assert occ_count > 50, f"Expected > 50 occupied cells, found {occ_count}."
assert free_count > 200, f"Expected > 200 free cells, found {free_count}."

# Test 4: Costmap Layer Distribution
lethal_count = np.sum(costmap == 254)
inscribed_count = np.sum(costmap == 253)
decay_count = np.sum((costmap >= 1) & (costmap <= 252))
free_cost_count = np.sum(costmap == 0)

print(
    f"Costmap Statistics: {lethal_count} lethal, {inscribed_count} inscribed, {decay_count} decay, {free_cost_count} free.")
assert lethal_count > 0, "No lethal cost cells found in costmap!"
assert inscribed_count > 0, "No inscribed safety margin cells found in costmap!"
assert decay_count > 0, "No decay inflation buffer cells found in costmap!"
assert free_cost_count > 0, "No zero-cost drivable cells found in costmap!"

print("✅ All Occupancy Grid & Costmap Inflation tests passed successfully!")
